# Ingeniería de datos con Databricks - Ingesta de datos en tiempo real para transacciones financieras

Construir un sistema en tiempo real que consuma mensajes de sistemas en vivo es necesario para crear aplicaciones de datos reactivas.

El procesamiento casi en tiempo real es clave para detectar nuevos patrones de fraude y construir un sistema proactivo, ofreciendo mejor protección a tus clientes.

Ingerir, transformar y limpiar datos para crear tablas SQL limpias para nuestros usuarios finales (Analistas de Datos y Científicos de Datos) es complejo.

<link href="https://fonts.googleapis.com/css?family=DM Sans" rel="stylesheet"/>
<div style="width:300px; text-align: center; float: right; margin: 30px 60px 10px 10px;  font-family: 'DM Sans'">
  <div style="height: 300px; width: 300px;  display: table-cell; vertical-align: middle; border-radius: 50%; border: 25px solid #fcba33ff;">
    <div style="font-size: 70px;  color: #70c4ab; font-weight: bold">
      73%
    </div>
    <div style="color: #1b5162;padding: 0px 30px 0px 30px;">de los datos empresariales no se utilizan para analítica y toma de decisiones</div>
  </div>
  <div style="color: #bfbfbf; padding-top: 5px">Fuente: Forrester</div>
</div>

<br>

## <img src="https://github.com/databricks-demos/dbdemos-resources/raw/main/images/de.png" style="float:left; margin: -35px 0px 0px 0px" width="80px"> John, como ingeniero de datos, dedica una enorme cantidad de tiempo…


* Programando a mano la ingesta y transformación de datos y enfrentando retos técnicos:<br>
  *Soportando streaming y batch, manejando operaciones concurrentes, problemas de archivos pequeños, requisitos GDPR, dependencias complejas de DAG...*<br><br>
* Construyendo frameworks personalizados para asegurar calidad y pruebas<br><br>
* Construyendo y manteniendo infraestructura escalable, con observabilidad y monitoreo<br><br>
* Gestionando modelos de gobernanza incompatibles de diferentes sistemas
<br style="clear: both">



<!-- Recopilar datos de uso (vista). Elimínalo para desactivar la recopilación. Consulta el README para más detalles.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=7474659742638075&notebook=%2F01-Data-ingestion%2F01.1-sdp-sql%2F01-SDP-fraud-detection-SQL&demo_name=lakehouse-fsi-fraud&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-fraud%2F01-Data-ingestion%2F01.1-sdp-sql%2F01-SDP-fraud-detection-SQL&version=1">

## Demo: construye una base de datos bancaria y detecta fraude en transacciones en tiempo real (ms)

En esta demo, nos pondremos en el lugar de una empresa bancaria minorista que procesa transacciones.

El negocio ha determinado que debemos mejorar nuestro sistema de detección de fraude en transacciones y ofrecer mejor protección a nuestros clientes (minoristas e instituciones que usan nuestros sistemas de pago). Se nos pide:

* Analizar y explicar las transacciones actuales: cuantificar el fraude, entender patrones y uso
* Construir un sistema proactivo para detectar fraude y servir predicciones en tiempo real (con latencias de ms)


### Qué construiremos

Para ello, construiremos una solución de extremo a extremo con Lakehouse. Para poder analizar y detectar fraude correctamente, nos centraremos principalmente en los datos transaccionales recibidos por nuestro sistema bancario.

A muy alto nivel, este es el flujo que implementaremos:

<img width="1000px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/lakehouse-fsi-fraud-overview-1.png" />

1. Ingerir y crear nuestra base de datos bancaria, con tablas fáciles de consultar en SQL
2. Asegurar los datos y conceder acceso de lectura a los equipos de Analistas y Científicos de Datos.
3. Ejecutar consultas BI para analizar el fraude existente
4. Construir un modelo de ML para detectar fraude y desplegar este modelo para inferencia en tiempo real

Como resultado, tendremos toda la información necesaria para activar alertas y pedir autenticación reforzada a nuestros clientes si creemos que hay alto riesgo de fraude.

**Nota sobre la detección de fraude en aplicaciones reales** <br/>
*Esta demo es un ejemplo simple para mostrar los beneficios de Lakehouse. Mantendremos el modelo de datos y ML simple por motivos demostrativos. Aplicaciones reales requerirían más fuentes de datos y tratarían con clases desbalanceadas y modelos más avanzados. Si te interesa una discusión más avanzada, ¡contacta a tu equipo de Databricks!*

¡Veamos cómo estos datos pueden usarse en Lakehouse para analizar y reducir el fraude!


## Construyendo un pipeline de Spark Declarative Pipelines para analizar y reducir la detección de fraude en tiempo real

En este ejemplo, implementaremos un pipeline SDP de extremo a extremo que consume la información de transacciones bancarias. Usaremos la arquitectura medallion, aunque podríamos construir un esquema estrella, un data vault u otro modelo.

Cargaremos datos nuevos de forma incremental con Autoloader y enriqueceremos esta información.

Esta información se usará para:

* Construir nuestro dashboard DBSQL para rastrear transacciones e impacto del fraude.
* Entrenar y desplegar un modelo para detectar posible fraude en tiempo real.

Implementemos el siguiente flujo:

<img width="1200px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-full.png"/>

*Nota que estamos incluyendo el modelo de ML que nuestro [Científico de Datos construyó]($../04-Data-Science-ML/04.1-AutoML-FSI-fraud) usando Databricks AutoML para predecir fraude. Cubriremos esto en la siguiente sección.*

¡Tu pipeline SDP ha sido instalado y arrancado por ti! Abre la <a dbdemos-pipeline-id="sdp-fsi-fraud" href="#joblist/pipelines/aea9c893-dbc5-4d79-93c2-f44617361b06" target="_blank">pipeline de Spark Declarative Pipelines para detección de fraude</a> para verlo en acción.<br/>
*(Nota: El pipeline arrancará automáticamente una vez que el trabajo de inicialización termine, esto puede tomar unos minutos... Revisa los logs de instalación para más detalles)*


## 1/ Exploración de Datos

Todos los proyectos de datos inician con algo de exploración. Abre el notebook [/explorations/sample_exploration]($./explorations/sample_exploration) para comenzar y descubrir los datos disponibles para ti.

### 2/ Cargando nuestros datos usando Databricks Autoloader (cloud_files)

<img  style="float:right; margin-left: 10px" width="600px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-1.png"/>

Autoloader nos permite ingerir millones de archivos desde un almacenamiento en la nube de manera eficiente, y soporta inferencia y evolución de esquemas a escala.

Para más detalles sobre autoloader, ejecuta `dbdemos.install('auto-loader')`.

Usemos esto en nuestro pipeline para ingerir los datos crudos en formato JSON y CSV que se entregan en nuestro blob storage `/dbdemos/fsi/fraud-detection/...`.

Abre el archivo [/transformations/01-bronze.sql]($./transformations/01-bronze.sql) para revisar el código SQL que ingiere los datos crudos y crea nuestra capa bronze.

### 3/ Asegura la calidad y materializa nuestras tablas para los Analistas de Datos

<img style="float:right; margin-left: 10px" width="600px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-2.png"/>

La siguiente capa, llamada Silver, consume datos **incrementales** de la bronze y limpia información:

* Limpia los códigos de país de origen y destino (eliminando los "--")
* Calcula la diferencia entre los saldos de origen y destino.

También agregamos una [expectativa](https://docs.databricks.com/workflows/delta-live-tables/delta-live-tables-expectations.html) en diferentes campos para asegurar y rastrear la calidad de los datos. Esto garantiza que nuestros dashboards sean relevantes y puedan detectar fácilmente errores por anomalías en los datos.

Para capacidades SDP más avanzadas ejecuta `dbdemos.install('pipeline-bike')` o `dbdemos.install('declarative-pipeline-cdc')` para un ejemplo de CDC/SCD Tipo 2.

¡Estas tablas están limpias y listas para ser usadas por el equipo de BI!

Abre el archivo [/transformations/02-silver.sql]($./transformations/02-silver.sql) para revisar el código SQL que crea nuestra tabla silver limpia.

### 4/ Agrega y une datos para crear nuestras variables de ML

<img style="float:right; margin-left: 10px" width="600px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/fraud-detection/fsi-fraud-dlt-3.png"/>

Ahora estamos listos para crear las variables necesarias para la detección de fraude.

Necesitamos enriquecer nuestro conjunto de transacciones con información extra que nuestro modelo usará para ayudar a detectar fraude.

Abre el archivo [/transformations/03-gold.sql]($./transformations/03-gold.sql) para revisar el código SQL que crea nuestra tabla gold enriquecida.

## ¡Nuestro pipeline ya está listo!

Como puedes ver, construir pipelines de datos con Databricks te permite enfocarte en la lógica de negocio mientras el motor resuelve todo el trabajo duro de ingeniería de datos.

La tabla ya está lista para que nuestro Científico de Datos entrene un modelo para detectar riesgo de fraude.

¡Abre la <a dbdemos-pipeline-id="sdp-fsi-fraud" href="#joblist/pipelines/aea9c893-dbc5-4d79-93c2-f44617361b06" target="_blank">pipeline de Spark Declarative Pipelines para detección de fraude</a> y haz clic en Start para visualizar tu linaje y consumir los nuevos datos incrementalmente!

# Siguiente: asegura y comparte datos con Unity Catalog

Ahora que estas tablas están disponibles en nuestro Lakehouse, revisemos cómo podemos compartirlas con los equipos de Científicos y Analistas de Datos.

Ve al notebook [Gobernanza con Unity Catalog]($../../02-Data-governance/02-UC-data-governance-ACL-fsi-fraud) o [Vuelve a la introducción]($../../00-FSI-fraud-detection-introduction-lakehouse)


### Datos de origen

Este conjunto de datos está construido con PaySim, un simulador de transacciones bancarias de código abierto.

[PaySim](https://github.com/EdgarLopezPhD/PaySim) simula transacciones de dinero móvil basadas en una muestra de transacciones reales extraídas de un mes de registros financieros de un servicio de dinero móvil implementado en un país africano.